# 03. Propensity Score Methods

## Background

When we can't randomize, we try to make the observed data look *as if* it were randomized — conditional on a set of pre-treatment covariates X. If we believe the **conditional ignorability** assumption (Y(0),Y(1) ⊥ T | X), then the causal effect is identified: within strata of X, treatment assignment is effectively random.

The problem is that X is usually high-dimensional. Conditioning on 20 covariates directly leads to the curse of dimensionality — tiny or empty cells. Rosenbaum & Rubin (1983) showed that you only need to condition on the **propensity score** e(X) = P(T=1|X) — a single scalar summary. If ignorability holds given X, it holds given e(X).

## What You'll Learn

- Propensity score estimation with logistic regression and gradient boosting
- Overlap / common support: the region where both treated and control units exist
- Propensity score matching: 1:1 nearest neighbor, caliper matching
- Trimming: dropping units outside the common support region
- Diagnosing balance *after* matching via love plots and SMDs

## Why This Matters

Propensity score matching is one of the most widely used observational methods in health economics, epidemiology, and policy evaluation (Dehejia & Wahba 1999, LaLonde 1986 revisited). The key insight — reduce all confounders to a single score — enabled practical implementation of ignorability-based methods before the era of Double ML. Even today, checking overlap and trimming are standard first steps before any observational analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def smd(x_t, x_c):
    """Standardized Mean Difference."""
    mu_t, mu_c = np.mean(x_t), np.mean(x_c)
    pooled_sd = np.sqrt((np.var(x_t, ddof=1) + np.var(x_c, ddof=1)) / 2)
    return (mu_t - mu_c) / (pooled_sd + 1e-10)


def love_plot(df, covariates, treatment_col='treated', title='', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 0.5 * len(covariates) + 1.5))
    smds = {c: smd(df.loc[df[treatment_col], c].values,
                   df.loc[~df[treatment_col], c].values) for c in covariates}
    cols = sorted(smds, key=lambda c: abs(smds[c]))
    vals = [smds[c] for c in cols]
    colors = ['#F44336' if abs(v) > 0.1 else '#4CAF50' for v in vals]
    ax.barh(cols, vals, color=colors, alpha=0.8)
    ax.axvline(0, color='black', lw=1)
    ax.axvline(0.1, color='orange', lw=1.5, ls='--')
    ax.axvline(-0.1, color='orange', lw=1.5, ls='--')
    ax.set_xlabel('SMD'); ax.set_title(title)
    return ax

## 1. Simulating a Confounded Observational Study

In [ ]:
def simulate_study(n=2000, seed=0):
    """
    DGP with 5 observed confounders and true ATE = 3.
    Selection is strongly confounded by age and severity.
    """
    rng = np.random.default_rng(seed)
    age      = rng.normal(50, 12, n)
    severity = rng.normal(5, 2, n)   # disease severity (higher = sicker)
    bmi      = rng.normal(27, 5, n)
    smoke    = rng.binomial(1, 0.3, n)
    female   = rng.binomial(1, 0.5, n)

    # Confounded assignment: sicker + older → more likely treated
    logit = -3 + 0.06*age + 0.4*severity - 0.02*bmi + 0.3*smoke
    ps_true = 1 / (1 + np.exp(-logit))
    treated = rng.binomial(1, ps_true).astype(bool)

    # Outcomes: higher severity → worse, treatment helps by 3 units
    y0 = 20 - 0.8*severity + 0.1*age - 0.2*bmi + rng.normal(0, 2, n)
    y1 = y0 + 3.0  # constant ATE = 3
    y_obs = np.where(treated, y1, y0)

    return pd.DataFrame({
        'treated': treated,
        'age': age, 'severity': severity, 'bmi': bmi,
        'smoke': smoke, 'female': female,
        'y_obs': y_obs, 'ps_true': ps_true,
        'y0': y0, 'y1': y1,
    })


df = simulate_study(n=2000)
cov_cols = ['age','severity','bmi','smoke','female']

dim_raw = df.loc[df.treated,'y_obs'].mean() - df.loc[~df.treated,'y_obs'].mean()
print(f"True ATE       = 3.000")
print(f"Naïve DiM      = {dim_raw:.3f}   (biased by confounding)")
print(f"Treated n={df.treated.sum()}, Control n={(~df.treated).sum()}")

## 2. Propensity Score Estimation

We model e(X) = P(T=1|X) using logistic regression or gradient boosting. The propensity score is a nuisance model — we care about balance, not predictive accuracy.

In [ ]:
# Estimate PS with logistic regression
X = df[cov_cols].values
y_treat = df['treated'].values.astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=500, random_state=0)
lr.fit(X_scaled, y_treat)
df['ps_lr'] = lr.predict_proba(X_scaled)[:, 1]

# Also fit gradient boosting (better for non-linear relationships)
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)
gb.fit(X_scaled, y_treat)
df['ps_gb'] = gb.predict_proba(X_scaled)[:, 1]

print(f"LR  PS: mean treated={df.loc[df.treated,'ps_lr'].mean():.3f}, "
      f"control={df.loc[~df.treated,'ps_lr'].mean():.3f}")
print(f"GB  PS: mean treated={df.loc[df.treated,'ps_gb'].mean():.3f}, "
      f"control={df.loc[~df.treated,'ps_gb'].mean():.3f}")

## 3. Overlap / Common Support

Before matching or weighting, check that treated and control units have similar PS distributions. Units in regions of poor overlap (PS near 0 or 1 for one group only) are extrapolating — their counterfactuals are imputed from very different units.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ps_col, label in zip(axes, ['ps_lr', 'ps_gb'], ['Logistic Regression', 'Gradient Boosting']):
    ax.hist(df.loc[df.treated, ps_col], bins=40, density=True, alpha=0.6,
            color='#F44336', label='Treated')
    ax.hist(df.loc[~df.treated, ps_col], bins=40, density=True, alpha=0.6,
            color='#2196F3', label='Control')
    ax.set_xlabel('Propensity Score P(T=1|X)')
    ax.set_ylabel('Density')
    ax.set_title(f'PS Distribution — {label}')
    ax.legend()
    # Mark common support region
    lo = df.loc[df.treated, ps_col].min()
    hi = df.loc[~df.treated, ps_col].max()
    ax.axvspan(lo, hi, alpha=0.05, color='green')

plt.suptitle('Overlap / Common Support Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_overlap.png', dpi=110, bbox_inches='tight')
plt.show()

# Identify non-overlap region
cs_lo = df.loc[df.treated, 'ps_lr'].min()
cs_hi = df.loc[~df.treated, 'ps_lr'].max()
outside = ((df['ps_lr'] < cs_lo) | (df['ps_lr'] > cs_hi)).sum()
print(f"Common support: [{cs_lo:.3f}, {cs_hi:.3f}]")
print(f"Units outside common support: {outside} ({outside/len(df):.1%})")

## 4. Nearest-Neighbor Matching

For each treated unit, find the control unit with the closest propensity score (1:1 matching without replacement). Matched controls serve as counterfactuals.

In [ ]:
def nn_matching(df, ps_col='ps_lr', caliper=None, seed=0):
    """
    1:1 nearest-neighbor PS matching without replacement.
    caliper: maximum PS distance allowed (None = no limit).
    Returns matched DataFrame.
    """
    rng = np.random.default_rng(seed)
    treated_idx = df.index[df['treated']].tolist()
    control_idx = df.index[~df['treated']].tolist()

    # Shuffle to avoid systematic bias in tie-breaking
    rng.shuffle(treated_idx)

    matched_pairs = []
    available_controls = set(control_idx)

    for t_idx in treated_idx:
        if not available_controls:
            break
        ps_t = df.loc[t_idx, ps_col]
        avail = list(available_controls)
        ps_c = df.loc[avail, ps_col].values
        dists = np.abs(ps_c - ps_t)
        best_idx = avail[np.argmin(dists)]
        best_dist = dists.min()

        if caliper is not None and best_dist > caliper:
            continue  # no match within caliper

        matched_pairs.append((t_idx, best_idx))
        available_controls.remove(best_idx)

    t_ids = [p[0] for p in matched_pairs]
    c_ids = [p[1] for p in matched_pairs]
    df_matched = pd.concat([df.loc[t_ids], df.loc[c_ids]])
    print(f"Matched {len(matched_pairs)} pairs "
          f"({len(matched_pairs)/df.treated.sum():.1%} of treated matched)")
    return df_matched, matched_pairs


df_matched, pairs = nn_matching(df, ps_col='ps_lr', caliper=0.05)

# ATT from matched sample
att_matched = df_matched.loc[df_matched.treated, 'y_obs'].mean() -               df_matched.loc[~df_matched.treated, 'y_obs'].mean()
print(f"\nTrue ATT = {df.loc[df.treated,'y1'].mean() - df.loc[df.treated,'y0'].mean():.3f}")
print(f"Matched ATT estimate = {att_matched:.3f}")

In [ ]:
# Balance before vs after matching
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
love_plot(df,         cov_cols, title='Before matching', ax=axes[0])
love_plot(df_matched, cov_cols, title='After PS matching (caliper=0.05)', ax=axes[1])
plt.tight_layout()
plt.savefig('03_balance.png', dpi=110, bbox_inches='tight')
plt.show()

print("\nSMDs before/after matching:")
print(f"{'Covariate':<12} {'Before':>8} {'After':>8}")
for c in cov_cols:
    b = smd(df.loc[df.treated,c].values, df.loc[~df.treated,c].values)
    a = smd(df_matched.loc[df_matched.treated,c].values, df_matched.loc[~df_matched.treated,c].values)
    flag = ' ✓' if abs(a) < 0.1 else ' ✗'
    print(f"{c:<12} {b:>+8.3f} {a:>+8.3f}{flag}")

## 5. Trimming for Common Support

An alternative to matching: keep only units within the common support region and re-weight. This is simpler and avoids the arbitrary choice of caliper.

In [ ]:
def trim_to_common_support(df, ps_col='ps_lr', trim_pct=0.01):
    """
    Trim units outside the common support region.
    trim_pct: fraction of each tail to drop (handles boundary units).
    """
    ps_t = df.loc[df['treated'], ps_col]
    ps_c = df.loc[~df['treated'], ps_col]

    lo = max(ps_t.quantile(trim_pct), ps_c.quantile(trim_pct))
    hi = min(ps_t.quantile(1 - trim_pct), ps_c.quantile(1 - trim_pct))

    df_trim = df[(df[ps_col] >= lo) & (df[ps_col] <= hi)].copy()
    dropped = len(df) - len(df_trim)
    print(f"Common support: [{lo:.3f}, {hi:.3f}] — dropped {dropped} units ({dropped/len(df):.1%})")
    return df_trim


df_trim = trim_to_common_support(df, ps_col='ps_lr')
dim_trim = df_trim.loc[df_trim.treated,'y_obs'].mean() - df_trim.loc[~df_trim.treated,'y_obs'].mean()
print(f"\nNaïve DiM (trimmed sample) = {dim_trim:.3f}")
print("(Trimming alone doesn't solve confounding — still need weighting or matching)")

# Summary comparison
print(f"\n{'Method':<30} {'Estimate':>10}")
print(f"{'True ATE':<30} {'3.000':>10}")
print(f"{'Naïve DiM (raw)':<30} {dim_raw:>10.3f}")
print(f"{'PS Matching ATT':<30} {att_matched:>10.3f}")